# 41 — Ventaja Cuántica: RCS, Boson Sampling y Límites del Simulador Clásico

**Objetivo:** Entender las propuestas experimentales de *quantum advantage*, las métricas de verificación y los límites reales de la simulación clásica.

## Historia
- **Sycamore 2019** (Arute et al., *Nature*): 53 qubits, depth 20 → 200 segundos en hardware cuántico vs ~10,000 años estimados en Summit.
- **Jiuzhang 2020** (Zhong et al., *Science*): 76 modos fotónicos, Boson Sampling gaussiano → 2.5×10¹⁴ muestras equivalentes en 200 s.
- **Refutaciones parciales**: Pan & Zhang (2022, *PRX*) simularon Sycamore en ~304 segundos con redes tensoriales contratadas en GPU.

**Métricas clave:**
- **XEB** (Cross-Entropy Benchmarking): mide correlación entre distribución real y la ideal.
- **Permanent** del interferómetro: cuantifica el boson sampling.
- **Bond dimension χ**: parámetro que limita la simulación MPS.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.stats import unitary_group
from qiskit import QuantumCircuit, transpile
from qiskit.quantum_info import Statevector
from qiskit_aer import AerSimulator
from qiskit_aer.noise import NoiseModel, depolarizing_error

rng = np.random.default_rng(42)

## Parte A — Random Circuit Sampling (RCS)

Un circuito cuántico aleatorio de $n$ qubits y profundidad $d$ produce una distribución de salida que es **anti-concentrada**: similar a la distribución de Porter-Thomas (exponencial). Verificar esto es el núcleo de XEB.

In [ ]:
def random_circuit(n_qubits: int, depth: int, seed: int = 42) -> QuantumCircuit:
    """Circuito aleatorio tipo Sycamore: capas de rotaciones 1Q + CX alternados."""
    rng_local = np.random.default_rng(seed)
    qc = QuantumCircuit(n_qubits)
    for d in range(depth):
        for q in range(n_qubits):
            gate = rng_local.choice(['h', 'rx', 'ry', 'rz'])
            angle = rng_local.uniform(0, 2 * np.pi)
            if gate == 'h':
                qc.h(q)
            elif gate == 'rx':
                qc.rx(angle, q)
            elif gate == 'ry':
                qc.ry(angle, q)
            else:
                qc.rz(angle, q)
        start = d % 2
        for q in range(start, n_qubits - 1, 2):
            qc.cx(q, q + 1)
    return qc


def xeb_score(probs_ideal: np.ndarray, probs_noisy: np.ndarray) -> float:
    """XEB = 2^n * E_noisy[p_ideal(x)] - 1. Perfecto=1, aleatorio=0."""
    n = int(np.round(np.log2(len(probs_ideal))))
    return float(2**n * np.sum(probs_ideal * probs_noisy) - 1)


# Circuito aleatorio n=4, depth=10
n_q, depth_q = 4, 10
qc_rcs = random_circuit(n_q, depth_q, seed=0)
sv_ideal = Statevector(qc_rcs)
probs_ideal = np.abs(sv_ideal.data) ** 2

# Verificación Porter-Thomas: distribución de probabilidades debe ser exponencial
print('=== Anti-concentración Porter-Thomas ===')
print(f'  Media:     {probs_ideal.mean():.6f}  (uniforme: {1/2**n_q:.6f})')
print(f'  Varianza:  {probs_ideal.var():.2e}  (PT: {1/4**n_q:.2e})')
print(f'  Max/uniforme: {probs_ideal.max()/(1/2**n_q):.2f}×')

# Simulación ruidosa
p_err = 0.01
nm = NoiseModel()
nm.add_all_qubit_quantum_error(depolarizing_error(p_err, 1), ['h', 'rx', 'ry', 'rz'])
nm.add_all_qubit_quantum_error(depolarizing_error(p_err * 5, 2), ['cx'])
sim = AerSimulator(noise_model=nm)

qc_meas = qc_rcs.copy()
qc_meas.measure_all()
qc_t = transpile(qc_meas, sim, optimization_level=0)
counts = sim.run(qc_t, shots=8192).result().get_counts()
total = sum(counts.values())
probs_noisy = np.zeros(2**n_q)
for k, v in counts.items():
    probs_noisy[int(k, 2)] = v / total

xeb = xeb_score(probs_ideal, probs_noisy)
print(f'\nXEB score (p_err={p_err}): {xeb:.4f}  (ideal=1, aleatorio=0)')

In [ ]:
# XEB vs nivel de ruido
p_vals = np.linspace(0, 0.15, 12)
xeb_vals = []
n_gates = n_q * depth_q  # número de puertas 1Q aproximado

for p in p_vals:
    if p == 0:
        xeb_vals.append(1.0)
        continue
    nm_s = NoiseModel()
    nm_s.add_all_qubit_quantum_error(depolarizing_error(p, 1), ['h', 'rx', 'ry', 'rz'])
    nm_s.add_all_qubit_quantum_error(depolarizing_error(min(5*p, 0.999), 2), ['cx'])
    sim_s = AerSimulator(noise_model=nm_s)
    c = sim_s.run(transpile(qc_meas, sim_s, optimization_level=0), shots=4096).result().get_counts()
    tot = sum(c.values())
    pn = np.zeros(2**n_q)
    for k, v in c.items():
        pn[int(k, 2)] = v / tot
    xeb_vals.append(xeb_score(probs_ideal, pn))

# Teoría: XEB ≈ exp(-2·p_eff·n_gates)
xeb_teoria = np.exp(-2 * p_vals * n_gates)

fig, axes = plt.subplots(1, 2, figsize=(12, 5))

axes[0].plot(p_vals * 100, xeb_vals, 'b-o', ms=6, lw=2, label='XEB experimental')
axes[0].plot(p_vals * 100, xeb_teoria, 'r--', lw=2, label=r'exp(-2p·N_gates) teórico')
axes[0].axhline(0, color='gray', ls=':', lw=1)
axes[0].set_xlabel('Error por puerta p (%)'); axes[0].set_ylabel('XEB score')
axes[0].set_title('Cross-Entropy Benchmarking vs ruido'); axes[0].legend(); axes[0].grid(alpha=0.3)

axes[1].bar(range(2**n_q), probs_ideal, alpha=0.6, label='Ideal', color='steelblue')
axes[1].bar(range(2**n_q), probs_noisy, alpha=0.5, label=f'Ruidoso (p={p_err})', color='tomato')
axes[1].axhline(1/2**n_q, color='k', ls='--', lw=1, label='Uniforme')
axes[1].set_xlabel('Estado de salida'); axes[1].set_ylabel('Probabilidad')
axes[1].set_title('Distribución ideal vs ruidosa'); axes[1].legend(fontsize=8); axes[1].grid(alpha=0.2)

plt.tight_layout(); plt.show()

## Parte B — Boson Sampling y el Permanente

El **boson sampling** requiere calcular el **permanente** de submatrices de un interferómetro unitario. El permanente es #P-difícil clásicamente (Aaronson & Arkhipov, 2011).

$$P(T|S) = \frac{|\text{perm}(U_{S,T})|^2}{s_1! \cdots s_m! \cdot t_1! \cdots t_m!}$$

In [ ]:
from math import factorial


def permanent_ryser(M: np.ndarray) -> complex:
    """Permanente por algoritmo de Ryser — O(2^n · n)."""
    n = M.shape[0]
    if n == 0:
        return 1.0
    total = 0.0 + 0j
    for S in range(1, 2**n):
        bits = [i for i in range(n) if (S >> i) & 1]
        row_sum = np.sum(M[:, bits], axis=1)
        total += (-1) ** (len(bits) + 1) * np.prod(row_sum)
    return (-1)**n * total


def boson_sampling_prob(U: np.ndarray, s_in: list, s_out: list) -> float:
    """Probabilidad de transición de s_in a s_out en interferómetro U."""
    rows = sum([[i] * k for i, k in enumerate(s_in)], [])
    cols = sum([[j] * k for j, k in enumerate(s_out)], [])
    if len(rows) != len(cols) or len(rows) == 0:
        return 0.0
    U_sub = U[np.ix_(rows, cols)]
    perm = permanent_ryser(U_sub)
    denom = 1
    for k in s_in:  denom *= factorial(k)
    for k in s_out: denom *= factorial(k)
    return abs(perm)**2 / denom


# Demo: 4 modos, 2 fotones
U_bs = unitary_group.rvs(4, random_state=rng)
s_in = [1, 1, 0, 0]  # fotones en modos 0 y 1

# Todas las distribuciones de salida posibles (2 fotones en 4 modos)
from itertools import combinations_with_replacement
outputs_2ph = []
for combo in combinations_with_replacement(range(4), 2):
    s = [0, 0, 0, 0]
    for c in combo:
        s[c] += 1
    if tuple(s) not in [tuple(o) for o in outputs_2ph]:
        outputs_2ph.append(s)

probs_bs = {}
for s_out in outputs_2ph:
    p = boson_sampling_prob(U_bs, s_in, s_out)
    probs_bs[tuple(s_out)] = p

total_prob = sum(probs_bs.values())
print('=== Boson Sampling: 4 modos, 2 fotones ===')
print(f'Suma de probabilidades: {total_prob:.6f}  (debe ser ~1.0)')
print('\nTop 5 resultados:')
for s, p in sorted(probs_bs.items(), key=lambda x: -x[1])[:5]:
    print(f'  {list(s)} → P = {p:.6f}')

# Comparativa coste clásico vs cuántico para el permanente
print('\n=== Coste computacional del permanente ===')
print(f'{"n fotones":>10} | {"ops Ryser":>12} | {"tiempo @ 1TFLOP (s)":>20}')
print('-' * 48)
for n_ph in [5, 10, 20, 30, 40, 50]:
    ops = 2**n_ph * n_ph
    t_s = ops / 1e12  # 1 TFLOP/s
    print(f'{n_ph:>10} | {ops:>12.2e} | {t_s:>20.2e}')

## Parte C — Límites de simulación clásica: MPS y redes tensoriales

La simulación MPS (Matrix Product State) representa el estado cuántico como:
$$|\psi\rangle = \sum_{i_1,\ldots,i_n} A^{i_1}_1 A^{i_2}_2 \cdots A^{i_n}_n |i_1 \cdots i_n\rangle$$

El parámetro **bond dimension χ** controla la cantidad de entrelazamiento representable. Un χ finito da error para estados muy entrelazados.

In [ ]:
def mps_cost(n_qubits: int, depth: int, chi: int) -> dict:
    """Coste estimado de simulación MPS."""
    flops = n_qubits * depth * chi**3
    mem_gb = n_qubits * chi**2 * 16 / 1e9  # bytes complex128
    return {'n': n_qubits, 'depth': depth, 'chi': chi,
            'GFLOP': flops / 1e9, 'RAM_GB': mem_gb}

print(f'{"n":>4} | {"χ":>6} | {"depth":>6} | {"GFLOP":>12} | {"RAM GB":>10}')
print('-' * 50)
for n_q in [20, 53, 76, 127]:
    for chi in [64, 512, 4096]:
        c = mps_cost(n_q, 20, chi)
        print(f"{n_q:>4} | {chi:>6} | {20:>6} | {c['GFLOP']:>12.2e} | {c['RAM_GB']:>10.4f}")
    print()

# Diagrama: n_qubits simulables exactamente vs chi
chi_vals = np.logspace(1, 5, 100)
n_max = 2 * np.log2(chi_vals)  # para estado maximalmente entrelazado

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

axes[0].semilogx(chi_vals, n_max, 'b-', lw=2.5, label='n exactamente simulable')
axes[0].axhline(53, color='r', ls='--', lw=1.5, label='Google Sycamore (n=53)')
axes[0].axhline(76, color='orange', ls='--', lw=1.5, label='Jiuzhang (n=76 modos)')
axes[0].axhline(127, color='purple', ls='--', lw=1.5, label='IBM Eagle (n=127)')
axes[0].fill_between(chi_vals, 0, n_max, alpha=0.1, color='blue')
axes[0].set_xlabel('Bond dimension χ (log scale)')
axes[0].set_ylabel('n qubits simulables exactamente')
axes[0].set_title('Límite de simulación MPS vs χ')
axes[0].legend(fontsize=9); axes[0].grid(alpha=0.3)
axes[0].set_ylim(0, 150)

# Tiempo de simulación vs n para GPU moderna (10 PFLOP/s)
n_vals = np.arange(10, 80)
chi_fixed = 512
t_mps = n_vals * 20 * chi_fixed**3 / 1e15  # segundos @ 1 PFLOP
t_sv  = 2.0**n_vals * 16 / (1e12 * 8)      # RAM-limited statevector @ 1 TB RAM

axes[1].semilogy(n_vals, t_mps, 'b-', lw=2, label=f'MPS (χ={chi_fixed}, depth=20)')
axes[1].semilogy(n_vals, np.maximum(t_sv, 1e-10), 'r--', lw=2, label='Statevector (RAM-limited)')
axes[1].axhline(200, color='gray', ls=':', lw=1, label='Sycamore experiment (200 s)')
axes[1].axhline(3.15e7, color='k', ls=':', lw=1, label='1 año')
axes[1].set_xlabel('n qubits'); axes[1].set_ylabel('Tiempo estimado (segundos)')
axes[1].set_title('Tiempo de simulación clásica')
axes[1].legend(fontsize=8); axes[1].grid(alpha=0.3, which='both')

plt.tight_layout(); plt.show()

In [ ]:
resumen = """
ESTADO DEL ARTE — VENTAJA CUÁNTICA (2025)
══════════════════════════════════════════

1. RANDOM CIRCUIT SAMPLING (Google Sycamore 2019)
   • 53 qubits, depth 20 → XEB demostrado
   • Refutación: Pan & Zhang (PRX 2022) → 304s en GPU con tensor network
   • Lesson: XEB no prueba ventaja si el simulador puede explotar estructura

2. BOSON SAMPLING (Jiuzhang 2020, 2021)
   • 76-144 modos fotónicos → sampling gaussiano
   • Más robusto: permanente #P-difícil incluso aproximar
   • Escalado: cada fotón adicional cuadra el coste clásico

3. VENTAJA CUÁNTICA PRÁCTICA (problema útil): AÚN NO DEMOSTRADA
   • Requiere: error 2Q < 0.1% (actual: 0.3-1%)
   • Requiere: profundidad > 1000 puertas (actual: ~100 útiles)
   • Requiere: QEC con overhead < 1000× (actual: ~1000×)

4. CAMINO HACIA VENTAJA PRÁCTICA
   Año 2025-2027: QEC lógico + algoritmos FT (QSVT, HHL, Grover)
   Año 2027-2030: Ventaja en química cuántica (Hamiltoniano Hubbard)
   Año 2030+:     Ventaja en optimización, ML, criptografía
"""
print(resumen)

## Referencias

- Arute et al., *Quantum supremacy using a programmable superconducting processor*, **Nature** 574, 505 (2019)
- Zhong et al., *Quantum computational advantage using photons*, **Science** 370, 1460 (2020)
- Pan & Zhang, *Simulating the Sycamore quantum supremacy circuits*, **PRX Quantum** 3, 010317 (2022)
- Aaronson & Arkhipov, *The computational complexity of linear optics*, **STOC** 2011
- Vidal, *Efficient classical simulation of slightly entangled quantum computations*, **PRL** 91, 147902 (2003)